In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import gc
import hashlib
import re
import unicodedata
from pathlib import Path
from decimal import Decimal
import json
import torch
import random

import pandas as pd
from peft import PeftModel
from PIL import Image
from collections import Counter
from safetensors.torch import load_file
from transformers import GenerationConfig, Qwen3VLForConditionalGeneration, AutoProcessor

In [ ]:
MLLM = Path("/content/drive/MyDrive/MLLM/mordern_llm")
D_ROOT = Path("/content/drive/MyDrive/MLLM/section_d")
D_DATA = D_ROOT / "data"
D_DATA.mkdir(parents=True, exist_ok=True)

C_DATA = MLLM / "data/sroie"
manifest = json.loads((C_DATA / "manifest.json").read_text(encoding="utf-8"))
task = json.loads((C_DATA / "task_config.json").read_text(encoding="utf-8"))

TRAIN = manifest["train"]
FIELDS = tuple(task["fields"])
DOC_PROMPT = task["prompt"]
IMAGE_KWARGS = task["image_kwargs"]

processor = AutoProcessor.from_pretrained(MLLM / "outputs/qlora_run1/best_adapter", use_fast=False)
processor.tokenizer.padding_side = "right"

print("Section C folder:", MLLM)
print("Train / validation:", len(TRAIN), len(manifest["val"]))
print("Fields:", FIELDS)

In [ ]:
def normal(field, value):
    if not isinstance(value, str) or not value.strip():
        return None

    text = " ".join(unicodedata.normalize("NFKC", value).casefold().split())
    if field != "total":
        return text
    
    text = re.sub(r"^(?:rm|\$)\s*", "", text)
    if not re.fullmatch(
        r"[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d+)?", text
    ):
        return None

    amount = Decimal(text.replace(",", ""))
    return format(amount.normalize(), "f")


def score_doc(answer, reference):
    raw_valid = True
    used_cleanup = False

    try:
        prediction = json.loads(answer)
    except json.JSONDecodeError:
        raw_valid = False
        prediction = None

        fence = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", answer.strip(), flags=re.IGNORECASE | re.DOTALL)

        if fence:
            used_cleanup = True
            try:
                prediction = json.loads(fence.group(1))
            except json.JSONDecodeError:
                pass

    schema_valid = (
        isinstance(prediction, dict)
        and set(prediction) == set(FIELDS)
        and all(value is None or isinstance(value, str) for value in prediction.values())
    )

    flags = {}

    for field in FIELDS:
        correct = False

        if schema_valid:
            predicted = prediction[field]
            expected = reference[field]

            if predicted is None or expected is None:
                correct = predicted is None and expected is None
            else:
                normalized = normal(field, predicted)
                correct = (
                    normalized is not None
                    and normalized == normal(field, expected)
                )

        flags[f"{field}_correct"] = correct

    return {
        "prediction": prediction,
        "raw_json_valid": raw_valid,
        "schema_valid": schema_valid,
        "used_fence_cleanup": used_cleanup,
        **flags,
        "all_correct": all(flags.values())
    }